# Extended tasks + batched evaluation of finished seeds

Runs on GPU T4 x2 with Internet. Tick the `HF_TOKEN` secret for this notebook (it reads the finished checkpoints and writes new results back).

1. **Check batching**: evaluates 40 tasks one-at-a-time and in batches of 16 on the same model, prints the speed-up and how many tasks disagree. The notebook stops here if fewer than 95% agree.
2. **Extended eval**: for every seed in `SEEDS`, restores that seed's checkpoints from Hugging Face and evaluates each on the new 158-task set (38 new templates, 12 tools). It also re-runs the original 121 tasks in batches, so the batched results can be compared with the recorded one-at-a-time ones, and keeps every raw conversation for failure analysis.

Each (seed, condition) is a resumable stage, so if the session stops you re-run the same notebook.

Set the seeds to process (only seeds whose training is complete on Hugging Face). One seed takes roughly 15-25 minutes if batching works.

In [ ]:
SEEDS = [0, 1, 2, 4]
BATCH = 16

In [ ]:
import base64
import os
import subprocess
import sys

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# Kaggle secrets (Add-ons -> Secrets), all optional:
#   GH_TOKEN  read access to the GitHub repo, needed only while the repo is private
#   HF_TOKEN  write access to a Hugging Face repo, needed only to resume across sessions
os.environ.setdefault('ADBENCH_HF_REPO', 'NahlaNabil/adbench-run')
os.environ.setdefault('ADBENCH_RUN_TAG', 'v1-fixed')
try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    for _name in ('GH_TOKEN', 'HF_TOKEN'):
        try:
            os.environ[_name] = _secrets.get_secret(_name)
        except Exception:
            pass
except Exception:
    pass


def git(*args, timeout=600):
    """Run git without ever prompting (a credentials prompt would hang an unattended run for
    hours). Uses GH_TOKEN when set; if that fails (revoked token, or a public repo that needs
    none) it retries once without it."""
    env = {**os.environ, 'GIT_TERMINAL_PROMPT': '0'}
    token = os.environ.get('GH_TOKEN')
    for use_token in ([True, False] if token else [False]):
        cmd = ['git']
        if use_token:
            basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
            cmd += ['-c', f'http.https://github.com/.extraheader=AUTHORIZATION: basic {basic}']
        try:
            subprocess.run(cmd + list(args), check=True, timeout=timeout, env=env)
            return
        except subprocess.CalledProcessError:
            if not use_token:
                raise
            print('git with GH_TOKEN failed; retrying without it.')


def run_module(*args, timeout=4 * 3600):
    """Run `python -m <args>` in a fresh process, print the tail of its output, and raise if it
    fails or exceeds `timeout` seconds (a bare `!` command never stops the notebook)."""
    proc = subprocess.run(
        [sys.executable, '-m', *args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, timeout=timeout
    )
    print(proc.stdout[-20000:])
    proc.check_returncode()


ON_KAGGLE = os.path.isdir('/kaggle')
REPO_DIR = '/kaggle/working/agentic-distillation-benchmark' if ON_KAGGLE else '/content/agentic-distillation-benchmark'

if not os.path.isdir(REPO_DIR):
    git('clone', 'https://github.com/Nahla-Nabil/agentic-distillation-benchmark.git', REPO_DIR)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True, timeout=1800)

src_path = os.path.join(REPO_DIR, 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)
os.environ['PYTHONPATH'] = src_path + os.pathsep + os.environ.get('PYTHONPATH', '')

In [ ]:
# re-sync to the latest commit
git('-C', REPO_DIR, 'checkout', '--', '.')
git('-C', REPO_DIR, 'pull')

## 1. Does batching reproduce one-at-a-time evaluation?

In [ ]:
run_module("adbench.training.train", "--condition", "base")

In [ ]:
proc = subprocess.run(
    [sys.executable, "scripts/check_batched_eval.py", "--condition", "base", "--n", "40", "--batch", str(BATCH)],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
print(proc.stdout[-6000:])
proc.check_returncode()   # stops the notebook if agreement is below 95%

## 2. Extended tasks on the finished seeds

In [ ]:
import gc
import json
import shutil

os.environ["ADBENCH_EVAL_BATCH"] = str(BATCH)
os.environ["ADBENCH_STORE_TRANSCRIPTS"] = "1"

from adbench import pipeline_state as ps
from adbench.evaluation.run_eval import agreement_with_recorded, reevaluate_condition
from adbench.training.train import REPO_ROOT, load_experiment_config

experiment_config = load_experiment_config("configs/experiment.yaml")
models_config = load_experiment_config("configs/models.yaml")


def free_gpu():
    gc.collect()
    try:
        import torch
        torch.cuda.empty_cache()
    except Exception:
        pass


summary = []
for seed in SEEDS:
    tag = f"v2-seed{seed}"
    os.environ["ADBENCH_RUN_TAG"] = tag
    # start every seed from a clean working copy so stage markers / checkpoints of the
    # previous seed can never be pushed under this seed's tag
    for d in ("checkpoints", "results"):
        shutil.rmtree(REPO_ROOT / d, ignore_errors=True)
    print(f"##### {tag} #####")
    if not ps.restore():
        print(f"{tag}: nothing on Hugging Face, skipping")
        continue
    ps.check_upload()

    recorded = [json.loads(line) for line in (REPO_ROOT / "results/eval_results.jsonl").read_text(encoding="utf-8").splitlines() if line.strip()]
    conditions = [c for c in ("base", "sft_only", "sft_early", "self_distill", "distilled")
                  if (REPO_ROOT / "checkpoints" / c).exists() and any(r["condition"] == c and r["task_set"] == "unseen_tools" for r in recorded)]
    for condition in conditions:
        stage = f"ext_eval_{condition}"
        cache_file = ps.stages_dir() / f"{stage}.json"
        if ps.is_done(stage) and cache_file.exists():
            rows = json.loads(cache_file.read_text(encoding="utf-8"))
            print(f"=== {tag} {condition}: restored ===")
        else:
            print(f"=== {tag}: extended eval of {condition} ===")
            rows = reevaluate_condition(condition, experiment_config, models_config, rerun_original=True)
            for row in rows:
                row["seed"] = seed
            ps.stages_dir().mkdir(parents=True, exist_ok=True)
            cache_file.write_text(json.dumps(rows), encoding="utf-8")
            ps.finish(stage)
            free_gpu()
        rerun = [r for r in rows if r["task_set"] == "unseen_tools_rerun"]
        old = [r for r in recorded if r["condition"] == condition and r["task_set"] == "unseen_tools"]
        agree = agreement_with_recorded(rerun, old)
        ext = [r for r in rows if r["task_set"] == "unseen_tools_ext"]
        summary.append({"seed": seed, "condition": condition, "ext_success": sum(r["success"] for r in ext) / max(len(ext), 1),
                        "rerun_success": sum(r["success"] for r in rerun) / max(len(rerun), 1), **agree})
        print(summary[-1])

In [ ]:
import pandas as pd

pd.set_option("display.width", 200)
df = pd.DataFrame(summary)
df

Agreement between the batched re-run and the recorded one-at-a-time rows, across everything just processed:

In [ ]:
print("tasks compared:", df["compared"].sum(),
      "| same success:", df["same_success"].sum(),
      "| identical step sequence:", df["same_steps"].sum())